In [1]:
# ============================================================
# EVI — Monitoramento da Vegetação 
# ============================================================
# Índice de Vegetação Melhorado (Enhanced Vegetation Index)
#
# Sentinel-2:
# B2 → Azul
# B4 → Vermelho
# B8 → Infravermelho próximo (NIR)
#
# Resolução: 10 m
# Período: 2025
# ============================================================

from pathlib import Path

import ee
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt

print("✓ Bibliotecas carregadas")

✓ Bibliotecas carregadas


In [2]:
# ============================================================
# 2. Pastas do projeto e Área de Interesse (AOI)
# ============================================================

# Pasta principal do projeto
pasta_projeto = Path.cwd().parent

# Pastas de dados
pasta_raw = pasta_projeto / "data" / "raw"
pasta_processed = pasta_projeto / "data" / "processed"

# Pasta específica do EVI
pasta_evi = pasta_processed / "evi"
pasta_evi.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Localiza automaticamente o shapefile da Área do Imóvel
# ------------------------------------------------------------

arquivos_imovel = list(
    pasta_raw.rglob("Area_do_Imovel.shp")
)

if not arquivos_imovel:
    raise FileNotFoundError(
        "❌ Area_do_Imovel.shp não encontrado dentro de data/raw"
    )

arquivo_imovel = arquivos_imovel[0]

print("✓ Pastas configuradas")
print(f"✓ Pasta EVI: {pasta_evi}")

print("\nÁrea do Imóvel:")
print(f"✓ Arquivo: {arquivo_imovel}")

# Leitura da camada
imovel = gpd.read_file(arquivo_imovel)

print(f"✓ CRS: {imovel.crs}")
print(f"✓ Feições: {len(imovel)}")

✓ Pastas configuradas
✓ Pasta EVI: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\evi

Área do Imóvel:
✓ Arquivo: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\raw\Area_do_Imovel\Area_do_Imovel.shp
✓ CRS: EPSG:4674
✓ Feições: 2


In [3]:
# ============================================================
# 3. Área de Interesse (AOI)
# ============================================================

# Os dois registros possuem a mesma geometria.
# Utilizamos apenas a primeira geometria como AOI.

aoi = imovel.geometry.iloc[0]

print("✓ AOI criada")
print(f"✓ Tipo de geometria: {aoi.geom_type}")
print(f"✓ CRS original: {imovel.crs}")

✓ AOI criada
✓ Tipo de geometria: Polygon
✓ CRS original: EPSG:4674


In [4]:
# ============================================================
# 3.1. Conversão da AOI para WGS84
# ============================================================

imovel_wgs84 = imovel.to_crs("EPSG:4326")

aoi_wgs84 = imovel_wgs84.geometry.iloc[0]

print("✓ AOI convertida para WGS84")
print(f"✓ CRS: {imovel_wgs84.crs}")
print(f"✓ Tipo: {aoi_wgs84.geom_type}")

✓ AOI convertida para WGS84
✓ CRS: EPSG:4326
✓ Tipo: Polygon


In [5]:
# ============================================================
# 4. Conexão com Google Earth Engine
# ============================================================

ee.Initialize()

print("✓ Google Earth Engine conectado")
# ============================================================
# Conversão da AOI para Google Earth Engine
# ============================================================

aoi_ee = ee.Geometry(aoi_wgs84.__geo_interface__)

print("✓ AOI convertida para Earth Engine")
print("✓ Tipo:", aoi_ee.type().getInfo())

✓ Google Earth Engine conectado
✓ AOI convertida para Earth Engine
✓ Tipo: Polygon


In [6]:
# ============================================================
# 5A. Definição automática do período de análise
# ============================================================

from datetime import datetime

# Ano da análise
ano = 2025
data_inicio = f"{ano}-01-01"
data_fim = f"{ano + 1}-01-01"

print("=" * 70)
print("PERÍODO DE ANÁLISE — SENTINEL-2")
print("=" * 70)

print(f"\n✓ Ano analisado: {ano}")
print(f"✓ Data inicial: {data_inicio}")
print(f"✓ Período: 1 ano")
print(f"✓ Data final calculada automaticamente: {data_fim}")

print("\n" + "=" * 70)
print("✓ PERÍODO DEFINIDO AUTOMATICAMENTE")
print("=" * 70)

PERÍODO DE ANÁLISE — SENTINEL-2

✓ Ano analisado: 2025
✓ Data inicial: 2025-01-01
✓ Período: 1 ano
✓ Data final calculada automaticamente: 2026-01-01

✓ PERÍODO DEFINIDO AUTOMATICAMENTE


In [7]:
# ============================================================
# 6. Busca da coleção Sentinel-2
# ============================================================

# Coleção Sentinel-2 Surface Reflectance Harmonized
colecao_s2 = "COPERNICUS/S2_SR_HARMONIZED"

# Limite máximo de cobertura de nuvens
limite_nuvens = 20

# Busca e filtra as imagens
colecao = (
    ee.ImageCollection(colecao_s2)
    .filterBounds(aoi_ee)
    .filterDate(
        data_inicio,
        data_fim
    )
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            limite_nuvens
        )
    )
)

# Quantidade de imagens encontradas
quantidade = colecao.size().getInfo()

print("=" * 70)
print("COLEÇÃO SENTINEL-2")
print("=" * 70)

print(f"\n✓ Coleção: {colecao_s2}")
print(f"✓ Período: {data_inicio} → {data_fim}")
print(f"✓ Limite de nuvens: < {limite_nuvens}%")
print(f"✓ Imagens encontradas: {quantidade}")

print("\n" + "=" * 70)
print("✓ COLEÇÃO SENTINEL-2 FILTRADA")
print("=" * 70)

COLEÇÃO SENTINEL-2

✓ Coleção: COPERNICUS/S2_SR_HARMONIZED
✓ Período: 2025-01-01 → 2026-01-01
✓ Limite de nuvens: < 20%
✓ Imagens encontradas: 99

✓ COLEÇÃO SENTINEL-2 FILTRADA


In [8]:
# ============================================================
# 7. Verificação da cobertura da AOI
# ============================================================
# Critério:
# Cobertura da AOI >= 99,99%
# ============================================================

def calcular_cobertura_aoi(imagem):
    """
    Calcula a porcentagem da AOI coberta
    por pixels válidos da imagem Sentinel-2.
    """

    mascara = imagem.select("B8").mask()

    cobertura = (
        mascara
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi_ee,
            scale=10,
            maxPixels=1e9
        )
        .get("B8")
    )

    return ee.Number(cobertura).multiply(100)


print("✓ Função de cobertura da AOI criada")

✓ Função de cobertura da AOI criada


In [9]:
# ============================================================
# 8. Seleção da melhor imagem de cada mês
# ============================================================

imagens_mensais = []

for mes in range(1, 13):

    inicio_mes = ee.Date.fromYMD(ano, mes, 1)   # ← ANO adicionado!
    fim_mes = inicio_mes.advance(1, "month")

    colecao_mes = (
        colecao
        .filterDate(inicio_mes, fim_mes)
    )

    # --------------------------------------------------------
    # Calcula a cobertura da AOI para cada imagem
    # --------------------------------------------------------

    def adicionar_cobertura(imagem):
        cobertura = calcular_cobertura_aoi(imagem)

        return imagem.set(
            "COBERTURA_AOI",
            cobertura
        )

    colecao_mes = colecao_mes.map(
        adicionar_cobertura
    )

    # --------------------------------------------------------
    # Primeiro: exige cobertura da AOI >= 99,99%
    # --------------------------------------------------------

    colecao_valida = colecao_mes.filter(
        ee.Filter.gte(
            "COBERTURA_AOI",
            99.99
        )
    )

    # --------------------------------------------------------
    # Segundo: seleciona a menor cobertura de nuvens
    # --------------------------------------------------------

    melhor = colecao_valida.sort(
        "CLOUDY_PIXEL_PERCENTAGE"
    ).first()

    imagens_mensais.append(melhor)

print("✓ Seleção mensal concluída")
print("✓ Critério 1: cobertura da AOI ≥ 99,99%")
print("✓ Critério 2: menor cobertura de nuvens")
print(f"✓ Meses processados: {len(imagens_mensais)}")

✓ Seleção mensal concluída
✓ Critério 1: cobertura da AOI ≥ 99,99%
✓ Critério 2: menor cobertura de nuvens
✓ Meses processados: 12


In [10]:
# ============================================================
# 9. Seleção mensal — cobertura completa da AOI
# ============================================================

# Ano da análise, obtido automaticamente da data inicial
ano = int(data_inicio[:4])

imagens_mensais = []

# Área total da AOI
area_aoi = aoi_ee.area(1)


# ------------------------------------------------------------
# Função para calcular a cobertura da imagem sobre a AOI
# ------------------------------------------------------------

def calcular_cobertura_aoi(imagem):

    intersecao = imagem.geometry().intersection(
        aoi_ee,
        1
    )

    area_intersecao = intersecao.area(1)

    cobertura = (
        area_intersecao
        .divide(area_aoi)
        .multiply(100)
    )

    return imagem.set(
        "COBERTURA_AOI",
        cobertura
    )


# ------------------------------------------------------------
# Seleção da melhor imagem de cada mês
# ------------------------------------------------------------

for mes in range(1, 13):

    inicio = f"{ano}-{mes:02d}-01"

    if mes == 12:
        fim = f"{ano + 1}-01-01"
    else:
        fim = f"{ano}-{mes + 1:02d}-01"


    # --------------------------------------------------------
    # Imagens do mês + cálculo da cobertura
    # --------------------------------------------------------

    colecao_mes = (
        colecao
        .filterDate(inicio, fim)
        .map(calcular_cobertura_aoi)
    )


    # --------------------------------------------------------
    # Mantém somente imagens que cobrem pelo menos
    # 99,99% da propriedade
    # --------------------------------------------------------

    colecao_completa = (
        colecao_mes
        .filter(
            ee.Filter.gte(
                "COBERTURA_AOI",
                99.99
            )
        )
        .sort(
            "CLOUDY_PIXEL_PERCENTAGE"
        )
    )


    quantidade = colecao_completa.size().getInfo()


    # --------------------------------------------------------
    # Nenhuma imagem adequada
    # --------------------------------------------------------

    if quantidade == 0:

        quantidade_total = colecao_mes.size().getInfo()

        if quantidade_total == 0:

            print(
                f"⚠️ {mes:02d}/{ano} — "
                "nenhuma imagem encontrada"
            )

        else:

            print(
                f"⚠️ {mes:02d}/{ano} — "
                "nenhuma imagem cobre 99,99% da AOI"
            )

        continue


    # --------------------------------------------------------
    # Melhor imagem do mês
    # --------------------------------------------------------

    imagem_mes = ee.Image(
        colecao_completa.first()
    )


    data = (
        ee.Date(
            imagem_mes.get(
                "system:time_start"
            )
        )
        .format("YYYY-MM-dd")
        .getInfo()
    )


    nuvens = imagem_mes.get(
        "CLOUDY_PIXEL_PERCENTAGE"
    ).getInfo()


    cobertura = imagem_mes.get(
        "COBERTURA_AOI"
    ).getInfo()


    tile = imagem_mes.get(
        "MGRS_TILE"
    ).getInfo()


    imagem_id = imagem_mes.get(
        "PRODUCT_ID"
    ).getInfo()


    # --------------------------------------------------------
    # Guarda informações da imagem
    # --------------------------------------------------------

    imagens_mensais.append({

        "mes": mes,

        "data": data,

        "nuvens": nuvens,

        "cobertura_aoi": cobertura,

        "tile": tile,

        "id": imagem_id,

        "imagem": imagem_mes

    })


    print(
        f"✓ {mes:02d}/{ano} | "
        f"{data} | "
        f"nuvens: {nuvens:.2f}% | "
        f"AOI: {cobertura:.2f}% | "
        f"tile: {tile}"
    )


print()
print(
    f"✓ Total de imagens selecionadas: "
    f"{len(imagens_mensais)}"
)

✓ 01/2025 | 2025-01-08 | nuvens: 3.87% | AOI: 100.00% | tile: 22KHV
✓ 02/2025 | 2025-02-17 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 03/2025 | 2025-03-09 | nuvens: 0.01% | AOI: 100.00% | tile: 22KHV
✓ 04/2025 | 2025-04-08 | nuvens: 0.02% | AOI: 100.00% | tile: 22KHV
✓ 05/2025 | 2025-05-13 | nuvens: 8.10% | AOI: 100.00% | tile: 23KKQ
✓ 06/2025 | 2025-06-17 | nuvens: 0.00% | AOI: 100.00% | tile: 23KKQ
✓ 07/2025 | 2025-07-27 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 08/2025 | 2025-08-01 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 09/2025 | 2025-09-15 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 10/2025 | 2025-10-05 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 11/2025 | 2025-11-21 | nuvens: 0.04% | AOI: 100.00% | tile: 22KHV
✓ 12/2025 | 2025-12-04 | nuvens: 1.61% | AOI: 100.00% | tile: 22KHV

✓ Total de imagens selecionadas: 12


In [11]:
# ============================================================
# 10. Cálculo do EVI
# ============================================================

def calcular_evi(imagem):

    # Sentinel-2 SR: conversão para reflectância
    azul = imagem.select("B2").multiply(0.0001)
    vermelho = imagem.select("B4").multiply(0.0001)
    nir = imagem.select("B8").multiply(0.0001)

    # Fórmula EVI
    evi = (
        nir.subtract(vermelho)
        .multiply(2.5)
        .divide(
            nir
            .add(vermelho.multiply(6))
            .subtract(azul.multiply(7.5))
            .add(1)
        )
        .rename("EVI")
    )

    return evi.copyProperties(
        imagem,
        [
            "system:time_start",
            "CLOUDY_PIXEL_PERCENTAGE",
            "COBERTURA_AOI"
        ]
    )

print("✓ Função EVI corrigida")

✓ Função EVI corrigida


In [12]:
# ============================================================
# 11. Processamento e organização dos produtos EVI
# ============================================================

evi_mensais = []

for item in imagens_mensais:

    # Recupera a imagem Sentinel-2 selecionada
    imagem = item["imagem"]

    # Recupera a data já armazenada no Passo 9
    data = item["data"]

    # Calcula o EVI com a função corrigida
    evi = calcular_evi(imagem)

    # Guarda o produto EVI e suas informações
    evi_mensais.append({
        "mes": item["mes"],
        "data": data,
        "nuvens": item["nuvens"],
        "cobertura_aoi": item["cobertura_aoi"],
        "tile": item["tile"],
        "id": item["id"],
        "evi": evi
    })


print("=" * 60)
print("PRODUTOS EVI PREPARADOS")
print("=" * 60)

for item in evi_mensais:

    print(
        f"✓ EVI | "
        f"{item['data']} | "
        f"nuvens: {item['nuvens']:.2f}% | "
        f"AOI: {item['cobertura_aoi']:.2f}%"
    )

print(
    f"\n✓ {len(evi_mensais)} produtos EVI preparados"
)

PRODUTOS EVI PREPARADOS
✓ EVI | 2025-01-08 | nuvens: 3.87% | AOI: 100.00%
✓ EVI | 2025-02-17 | nuvens: 0.00% | AOI: 100.00%
✓ EVI | 2025-03-09 | nuvens: 0.01% | AOI: 100.00%
✓ EVI | 2025-04-08 | nuvens: 0.02% | AOI: 100.00%
✓ EVI | 2025-05-13 | nuvens: 8.10% | AOI: 100.00%
✓ EVI | 2025-06-17 | nuvens: 0.00% | AOI: 100.00%
✓ EVI | 2025-07-27 | nuvens: 0.00% | AOI: 100.00%
✓ EVI | 2025-08-01 | nuvens: 0.00% | AOI: 100.00%
✓ EVI | 2025-09-15 | nuvens: 0.00% | AOI: 100.00%
✓ EVI | 2025-10-05 | nuvens: 0.00% | AOI: 100.00%
✓ EVI | 2025-11-21 | nuvens: 0.04% | AOI: 100.00%
✓ EVI | 2025-12-04 | nuvens: 1.61% | AOI: 100.00%

✓ 12 produtos EVI preparados


In [13]:
# ============================================================
# 12. Exportação dos EVI para a pasta local do projeto
#     (VERSÃO CORRIGIDA)
# ============================================================

from pathlib import Path
import requests

# Pasta local dos EVI
pasta_evi = Path(
    r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao"
) / "data" / "processed" / "indices" / "evi"

pasta_evi.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXPORTAÇÃO DOS EVI — ARQUIVOS LOCAIS (CORRIGIDA)")
print("=" * 60)

arquivos_baixados = []

for item in evi_mensais:

    data = item["data"].replace("-", "")
    nome = f"evi_{data}"
    arquivo_saida = pasta_evi / f"{nome}.tif"

    print(f"\nProcessando: {nome}")

    # Converte para ee.Image
    imagem_evi = ee.Image(item["evi"])
    
    # ✅ CORREÇÃO: aplicar clip na AOI ANTES do download
    imagem_evi_clip = imagem_evi.clip(aoi_ee)

    # URL de download direto do Earth Engine
    url = imagem_evi_clip.getDownloadURL({   # ← usa a imagem recortada
        "name": nome,
        "scale": 10,
        "region": aoi_ee,
        "filePerBand": False,
        "format": "GEO_TIFF"
    })

    resposta = requests.get(url)
    resposta.raise_for_status()

    with open(arquivo_saida, "wb") as arquivo:
        arquivo.write(resposta.content)

    arquivos_baixados.append(arquivo_saida)
    print(f"✓ Salvo: {arquivo_saida.name}")


print("\n" + "=" * 60)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 60)
print(f"\n✓ Total de arquivos: {len(arquivos_baixados)}")
print("✓ Pasta:")
print(pasta_evi)

EXPORTAÇÃO DOS EVI — ARQUIVOS LOCAIS (CORRIGIDA)

Processando: evi_20250108
✓ Salvo: evi_20250108.tif

Processando: evi_20250217
✓ Salvo: evi_20250217.tif

Processando: evi_20250309
✓ Salvo: evi_20250309.tif

Processando: evi_20250408
✓ Salvo: evi_20250408.tif

Processando: evi_20250513
✓ Salvo: evi_20250513.tif

Processando: evi_20250617
✓ Salvo: evi_20250617.tif

Processando: evi_20250727
✓ Salvo: evi_20250727.tif

Processando: evi_20250801
✓ Salvo: evi_20250801.tif

Processando: evi_20250915
✓ Salvo: evi_20250915.tif

Processando: evi_20251005
✓ Salvo: evi_20251005.tif

Processando: evi_20251121
✓ Salvo: evi_20251121.tif

Processando: evi_20251204
✓ Salvo: evi_20251204.tif

EXPORTAÇÃO CONCLUÍDA

✓ Total de arquivos: 12
✓ Pasta:
C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\evi
